# Pretrained Attention-Only LLM + Full SSONN Adapter

This notebook is a simple experiment with a pretrained attention-only TransformerLens model.

Unlike the previous sweep, the SSONN block is not added as a residual delta. Each attention block is wrapped as:

```python
resid_post = self.base_block(resid, *args, **kwargs)
return self.delta(self.ln_delta(resid_post))
```

The pretrained attention block is not frozen. The SSONN adapter is inserted after attention and is trained with the current `train_ssonn` setup used in the CIFAR notebook.

In [ ]:
# If needed, install the extra LLM dependencies first:
# %pip install transformer-lens datasets

In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
import torch.nn as nn
import torch.optim as optim
from datasets import load_dataset

from transformer_lens import HookedTransformer

import ssonn.train.train as ssonn_train
from ssonn.metrics.nonlinearity_metrics import AbsGradientEdgeMetric
from ssonn.model.model import ScaledNet

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
device

## Data

Use the same dataset family as `3b76cf790_ssonn_sweep_runner.py`: WikiText tokenized by the pretrained attention-only model tokenizer.

The only simplification is the target format. The library `train_ssonn` expects classification logits shaped `[batch, classes]`, so each sample is:

```text
input:  seq_len context tokens
target: next token after that context
```

In [ ]:
MODEL_NAME = "attn-only-1l"
DATASET_NAME = "wikitext"
DATASET_CONFIG = "wikitext-103-v1"
SEQ_LEN = 128
BATCH_SIZE = 4
TRAIN_BATCHES = 128
VAL_BATCHES = 24
FINDER_BATCHES = 6

base = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
tokenizer = base.tokenizer
base.cfg.d_model, base.cfg.n_layers, base.cfg.d_vocab

In [ ]:
def encode_text(text):
    if not text:
        return []
    try:
        ids = tokenizer.encode(text, add_special_tokens=False)
    except TypeError:
        ids = tokenizer.encode(text)
    eos_id = getattr(tokenizer, "eos_token_id", None)
    if eos_id is not None:
        ids = list(ids) + [int(eos_id)]
    return list(ids)


def make_next_token_batches(split, n_batches, shuffle=False, seed=0):
    dataset = load_dataset(DATASET_NAME, DATASET_CONFIG, split=split)
    if shuffle:
        dataset = dataset.shuffle(seed=seed)

    token_buffer = []
    batches = []
    block = SEQ_LEN + 1
    need = BATCH_SIZE * block

    for row in dataset:
        token_buffer.extend(encode_text(row.get("text", "")))
        while len(token_buffer) >= need and len(batches) < n_batches:
            batch = torch.tensor(token_buffer[:need], dtype=torch.long).view(BATCH_SIZE, block)
            del token_buffer[:need]
            x = batch[:, :-1].contiguous()
            y = batch[:, -1].contiguous()
            batches.append((x, y))
        if len(batches) >= n_batches:
            break

    if len(batches) < n_batches:
        raise RuntimeError(f"Only built {len(batches)} batches for split={split}; requested {n_batches}.")
    return batches


train_loader = make_next_token_batches("train", TRAIN_BATCHES, shuffle=True, seed=0)
val_loader = make_next_token_batches("validation", VAL_BATCHES, shuffle=False, seed=100)
finder_loader = make_next_token_batches("validation", FINDER_BATCHES, shuffle=False, seed=200)

del base
if torch.cuda.is_available():
    torch.cuda.empty_cache()

len(train_loader), len(val_loader), len(finder_loader), train_loader[0][0].shape, train_loader[0][1].shape

In [ ]:
def flatten_targets(batches):
    return torch.cat([y.cpu() for _, y in batches])


train_targets = flatten_targets(train_loader)
val_targets = flatten_targets(val_loader)
print("train examples:", int(train_targets.numel()))
print("val examples:", int(val_targets.numel()))
print("unique train targets:", int(train_targets.unique().numel()))
print("unique val targets:", int(val_targets.unique().numel()))
print("most common train target ratio:", float(torch.bincount(train_targets).max() / train_targets.numel()))
print("most common val target ratio:", float(torch.bincount(val_targets).max() / val_targets.numel()))
print("shared target ids:", int(torch.isin(val_targets.unique(), train_targets.unique()).sum()))

## Model

The attention block remains inside the model and stays trainable. The SSONN adapter replaces the block output, not adds a residual delta.

In [ ]:
class TokenSSONNAdapter(nn.Module):
    def __init__(self, d_model, device):
        super().__init__()
        weight = torch.eye(d_model, device=device)
        bias = torch.zeros(d_model, device=device)
        self.scaled = ScaledNet(weight, bias, device=str(device))
        self.scaled.dropout.p = 0.0

    def forward(self, x):
        batch, seq, width = x.shape
        y = self.scaled(x.reshape(batch * seq, width))
        return y.reshape(batch, seq, width)


class AttentionBlockWithFullSSONNAdapter(nn.Module):
    def __init__(self, base_block, d_model, adapter):
        super().__init__()
        self.base_block = base_block
        self.ln_delta = nn.LayerNorm(d_model)
        self.delta = adapter

    def forward(self, resid, *args, **kwargs):
        resid_post = self.base_block(resid, *args, **kwargs)
        return self.delta(self.ln_delta(resid_post))


class AttentionOnlyLMWithSSONNAdapters(nn.Module):
    def __init__(self, model_name, device):
        super().__init__()
        self.backbone = HookedTransformer.from_pretrained(model_name, device=device)
        d_model = int(self.backbone.cfg.d_model)

        self.ssonn_layer_names = []
        for idx, block in enumerate(self.backbone.blocks):
            adapter = TokenSSONNAdapter(d_model, device)
            layer_name = f"ssonn_adapter_{idx}"
            setattr(self, layer_name, adapter.scaled)
            self.ssonn_layer_names.append(layer_name)
            self.backbone.blocks[idx] = AttentionBlockWithFullSSONNAdapter(block, d_model, adapter).to(device)

    def forward(self, tokens):
        logits = self.backbone(tokens)
        return logits[:, -1, :]


model = AttentionOnlyLMWithSSONNAdapters(MODEL_NAME, device).to(device)

# Attention is not frozen: every parameter remains trainable.
for parameter in model.parameters():
    parameter.requires_grad = True

model.ssonn_layer_names

## Initial Weight Mode

Use `INITIAL_WEIGHT_MODE = "full_reinit"` if the LLM should be trained from random transformer-style initialization instead of pretrained weights. This is different from CNN-style Kaiming init: TransformerLens attention parameters are mostly raw `nn.Parameter`s, not `nn.Linear` modules, so `model.apply(init_weights)` would miss important matrices and use the wrong distribution.

In [ ]:
INITIAL_WEIGHT_MODE = "full_reinit"  # "pretrained" or "full_reinit"
TRANSFORMER_INIT_STD = 0.02


def reset_ssonn_adapters_to_identity(model):
    for name in model.ssonn_layer_names:
        layer = getattr(model, name)
        with torch.no_grad():
            layer.weight.zero_()
            diagonal = min(layer.out_features, layer.in_features)
            layer.weight[:diagonal, :diagonal] = torch.eye(
                diagonal, device=layer.weight.device, dtype=layer.weight.dtype
            )
            layer.bias.zero_()


def reinitialize_transformer_backbone(model, std=0.02):
    with torch.no_grad():
        for name, param in model.backbone.named_parameters():
            if not param.requires_grad:
                continue
            if param.ndim == 1 or name.split(".")[-1].startswith("b_") or name.endswith(".b"):
                param.zero_()
            else:
                nn.init.normal_(param, mean=0.0, std=std)

    for module in model.modules():
        if isinstance(module, nn.LayerNorm):
            with torch.no_grad():
                if module.elementwise_affine:
                    module.weight.fill_(1.0)
                    module.bias.zero_()


if INITIAL_WEIGHT_MODE == "pretrained":
    print("Using pretrained HookedTransformer weights.")
elif INITIAL_WEIGHT_MODE == "full_reinit":
    reinitialize_transformer_backbone(model, std=TRANSFORMER_INIT_STD)
    reset_ssonn_adapters_to_identity(model)
    print(f"Reinitialized HookedTransformer backbone with std={TRANSFORMER_INIT_STD}.")
else:
    raise ValueError(f"Unknown INITIAL_WEIGHT_MODE: {INITIAL_WEIGHT_MODE}")

## SSONN Training Setup

This follows the same pattern as `experiments/training/cifar10.ipynb`: define `hyperparams`, an `optimizer_factory`, then call `train_ssonn`.

There are two useful reinitialization modes after expansion:

- `REINIT_MODE = "ssonn_only"`: reinitialize only `ScaledNet`, leaving the current attention/backbone weights untouched.
- `REINIT_MODE = "full"`: reinitialize the whole HookedTransformer backbone with the same transformer-style initialization used by `INITIAL_WEIGHT_MODE = "full_reinit"`, then reset SSONN adapters to identity.

Run the notebook twice with different `REINIT_MODE` values to compare both hypotheses.

In [ ]:
REINIT_MODE = "ssonn_only"  # "ssonn_only" or "full"


def ssonn_only_weights_init(module):
    if isinstance(module, ScaledNet):
        module.reinitialize_parameters()


def transformer_full_weights_init(module):
    if isinstance(module, AttentionOnlyLMWithSSONNAdapters):
        reinitialize_transformer_backbone(module, std=TRANSFORMER_INIT_STD)
        reset_ssonn_adapters_to_identity(module)


if REINIT_MODE == "ssonn_only":
    ssonn_train.weights_init = ssonn_only_weights_init
elif REINIT_MODE == "full":
    ssonn_train.weights_init = transformer_full_weights_init
else:
    raise ValueError(f"Unknown REINIT_MODE: {REINIT_MODE}")

print("Reinitialization mode:", REINIT_MODE)

criterion = nn.CrossEntropyLoss()

hyperparams = {
    "num_epochs": 8,
    "batch_size": BATCH_SIZE,
    "edge_importance_metric": AbsGradientEdgeMetric(criterion),
    "edge_score_aggregation": "mean",
    "expansion_thresholds": {name: 0.85 for name in model.ssonn_layer_names},
    "pruning_thresholds": {name: 1.0 for name in model.ssonn_layer_names},
    "plateau_threshold": 0.01,
    "min_epochs_between_expansions": 1000000,
    "plateau_window_size": 5,
    "learning_rate": 2e-5,
    "prune_after_epochs": 4,
    "task_type": "classification",
    "start_fully_connected": False,
    "max_new_edges_per_expansion": 64,
    "weight_decay": 0.0,
    "diff_threshold": 0.05,
    "t_max": 2,
}


def optimizer_factory(model_instance, current_lr=None, base_lr=None):
    if base_lr is None:
        base_lr = hyperparams["learning_rate"]
    if current_lr is None:
        current_lr = base_lr

    optimizer = optim.AdamW(
        model_instance.parameters(),
        lr=current_lr,
        weight_decay=hyperparams["weight_decay"],
    )
    for group in optimizer.param_groups:
        group["initial_lr"] = base_lr
    return optimizer


hyperparams

## Comet Logging

Set `COMET_API_KEY`, `COMET_WORKSPACE`, and optionally `COMET_PROJECT_NAME` in the environment to enable live logging.

In [ ]:
experiment = None

if os.environ.get("COMET_API_KEY") and os.environ.get("COMET_WORKSPACE"):
    from comet_ml import start

    experiment = start(
        api_key=os.environ.get("COMET_API_KEY"),
        workspace=os.environ.get("COMET_WORKSPACE"),
        project_name=os.environ.get("COMET_PROJECT_NAME", "ssonn-experiments"),
    )
    experiment.set_name(
        f"attention_full_ssonn_{MODEL_NAME}_reinit_{REINIT_MODE}_seq{SEQ_LEN}_seed0"
    )
    experiment.log_parameters(
        {
            **hyperparams,
            "model_name": MODEL_NAME,
            "dataset_name": DATASET_NAME,
            "dataset_config": DATASET_CONFIG,
            "seq_len": SEQ_LEN,
            "train_batches": TRAIN_BATCHES,
            "val_batches": VAL_BATCHES,
            "finder_batches": FINDER_BATCHES,
            "reinit_mode": REINIT_MODE,
            "initial_weight_mode": INITIAL_WEIGHT_MODE,
            "transformer_init_std": TRANSFORMER_INIT_STD,
            "ssonn_layer_names": model.ssonn_layer_names,
        }
    )
else:
    print("Comet logging disabled: set COMET_API_KEY and COMET_WORKSPACE to enable it.")

experiment

### Metric sanity note

`train_accuracy` from `train_ssonn` is measured on the same fixed training batches that are optimized every epoch. With `TRAIN_BATCHES = 128` and `BATCH_SIZE = 4`, that is only 512 supervised next-token examples, while attention is fully trainable. Reaching ~100% train accuracy with low validation accuracy usually means memorization, not necessarily a target leak. Increase `TRAIN_BATCHES` or freeze part of the backbone if you want a harder generalization check.

In [ ]:
trained_model = ssonn_train.train_ssonn(
    model,
    train_loader,
    val_loader,
    finder_loader,
    criterion,
    optimizer_factory,
    hyperparams,
    device,
    experiment=experiment,
    use_log_tool=experiment is not None,
)

In [ ]:
for name in trained_model.ssonn_layer_names:
    layer = getattr(trained_model, name)
    print(name, "in_features=", layer.in_features, "sub_layers=", len(layer.sub_linears))